In [8]:
import requests

url = "https://npiregistry.cms.hhs.gov/api/"
params = {
    "version": "2.1",
    "postal_code": "19104",
    "limit": 200
}

response = requests.get(url, params=params)
data = response.json()

In [9]:
print(data.keys())

dict_keys(['result_count', 'results'])


In [11]:
taxonomy_list = []

for provider in data['results']:

    npi = provider.get('number')

    addresses = provider.get('addresses', [])

    street = None
    street2 = None
    city = None
    state = None
    zip_code = None

    if len(addresses) > 0:
        address = addresses[0]
        street = address.get('address_1')
        street2 = address.get('address_2')
        city = address.get('city')
        state = address.get('state')

        postal_code = address.get('postal_code')
        if postal_code is not None:
            zip_code = postal_code[:5]

    for taxonomy in provider.get('taxonomies', []):

        desc = taxonomy.get('desc')
        code = taxonomy.get('code')

        if desc is not None:
            parts = desc.split(",")

            category = parts[0].strip()
            specialty = parts[1].strip() if len(parts) > 1 else "None"

            taxonomy_list.append({
                "NPI": npi,
                "Code": code,
                "Category": category,
                "Specialty": specialty,
                "Street": street,
                "Street2": street2,
                "City": city,
                "State": state,
                "ZIP": zip_code
            })

import pandas as pd

provider_table = pd.DataFrame(taxonomy_list)
provider_table.head(5)

,NPI,Code,Category,Specialty,Street,Street2,City,State,ZIP
0,1790565554,314000000X,Skilled Nursing Facility,None,4401 HAVERFORD AVE,None,PHILADELPHIA,PA,19104
1,1255652087,207R00000X,Internal Medicine,None,3400 CIVIC CENTER BLVD,"DIVISION OF HEMATOLOGY-ONCOLOGY, PCAM 7, SPE",PHILADELPHIA,PA,19104
2,1255652087,390200000X,Student in an Organized Health Care Education/...,None,3400 CIVIC CENTER BLVD,"DIVISION OF HEMATOLOGY-ONCOLOGY, PCAM 7, SPE",PHILADELPHIA,PA,19104
3,1255652087,207RH0003X,Internal Medicine,Hematology & Oncology,3400 CIVIC CENTER BLVD,"DIVISION OF HEMATOLOGY-ONCOLOGY, PCAM 7, SPE",PHILADELPHIA,PA,19104
4,1598152944,2084N0400X,Psychiatry & Neurology,Neurology,330 S 9TH ST,None,PHILADELPHIA,PA,19107


In [12]:
import pandas as pd

provider_table = pd.DataFrame(taxonomy_list)
provider_table.head(10)

,NPI,Code,Category,Specialty,Street,Street2,City,State,ZIP
0,1790565554,314000000X,Skilled Nursing Facility,None,4401 HAVERFORD AVE,None,PHILADELPHIA,PA,19104
1,1255652087,207R00000X,Internal Medicine,None,3400 CIVIC CENTER BLVD,"DIVISION OF HEMATOLOGY-ONCOLOGY, PCAM 7, SPE",PHILADELPHIA,PA,19104
2,1255652087,390200000X,Student in an Organized Health Care Education/...,None,3400 CIVIC CENTER BLVD,"DIVISION OF HEMATOLOGY-ONCOLOGY, PCAM 7, SPE",PHILADELPHIA,PA,19104
3,1255652087,207RH0003X,Internal Medicine,Hematology & Oncology,3400 CIVIC CENTER BLVD,"DIVISION OF HEMATOLOGY-ONCOLOGY, PCAM 7, SPE",PHILADELPHIA,PA,19104
4,1598152944,2084N0400X,Psychiatry & Neurology,Neurology,330 S 9TH ST,None,PHILADELPHIA,PA,19107
5,1346860350,261QM0855X,Clinic/Center,Adolescent and Children Mental Health,954 BELMONT AVE,None,PHILADELPHIA,PA,19104
6,1891923306,2080P0204X,Pediatrics,Pediatric Emergency Medicine,100 E PENN SQ,9TH FL,PHILADELPHIA,PA,19107
7,1558005157,390200000X,Student in an Organized Health Care Education/...,None,3600 SPRUCE STREET,"4TH FLOOR, MALONEY BUILDING",PHILADELPHIA,PA,19104
8,1558005157,390200000X,Student in an Organized Health Care Education/...,None,3600 SPRUCE STREET,"4TH FLOOR, MALONEY BUILDING",PHILADELPHIA,PA,19104
9,1396780789,208000000X,Pediatrics,None,100 E PENN SQ,9TH FLOOR,PHILADELPHIA,PA,19107


In [13]:
import pandas as pd

nucc_df = pd.read_csv("data/nucc_taxonomy_251.csv")

provider_rows = []

for provider in data['results']:

    npi = provider.get('number')

    addresses = provider.get('addresses', [])
    city = None
    state = None
    zip_code = None
    address_1 = None
    address_2 = None

    if len(addresses) > 0:
        address = addresses[0]
        address_1 = address.get('address_1')
        address_2 = address.get('address_2')
        city = address.get('city')
        state = address.get('state')
        zip_code = address.get('postal_code')

    for taxonomy in provider.get('taxonomies', []):

        code = taxonomy.get('code')
        desc = taxonomy.get('desc')

        if desc is not None:
            parts = desc.split(",")
            category = parts[0].strip()
            specialty = parts[1].strip() if len(parts) > 1 else "None"
        else:
            category = None
            specialty = None

        provider_rows.append({
            "NPI": npi,
            "Code": code,
            "API_Category": category,
            "API_Specialty": specialty,
            "Address_1": address_1,
            "Address_2": address_2,
            "City": city,
            "State": state,
            "ZIP": zip_code
        })

provider_table = pd.DataFrame(provider_rows)

provider_table["ZIP"] = provider_table["ZIP"].astype(str).str[:5]

nucc_small = nucc_df[[
    "Code",
    "Grouping",
    "Classification",
    "Specialization",
    "Display Name"
]]

full_table = provider_table.merge(
    nucc_small,
    on="Code",
    how="left"
)

full_table.head(5)

,NPI,Code,API_Category,API_Specialty,Address_1,Address_2,City,State,ZIP,Grouping,Classification,Specialization,Display Name
0,1790565554,314000000X,Skilled Nursing Facility,None,4401 HAVERFORD AVE,None,PHILADELPHIA,PA,19104,Nursing & Custodial Care Facilities,Skilled Nursing Facility,NaN,Skilled Nursing Facility
1,1255652087,207R00000X,Internal Medicine,None,3400 CIVIC CENTER BLVD,"DIVISION OF HEMATOLOGY-ONCOLOGY, PCAM 7, SPE",PHILADELPHIA,PA,19104,Allopathic & Osteopathic Physicians,Internal Medicine,NaN,Internal Medicine Physician
2,1255652087,390200000X,Student in an Organized Health Care Education/...,None,3400 CIVIC CENTER BLVD,"DIVISION OF HEMATOLOGY-ONCOLOGY, PCAM 7, SPE",PHILADELPHIA,PA,19104,"Student, Health Care",Student in an Organized Health Care Education/...,NaN,Student in an Organized Health Care Education/...
3,1255652087,207RH0003X,Internal Medicine,Hematology & Oncology,3400 CIVIC CENTER BLVD,"DIVISION OF HEMATOLOGY-ONCOLOGY, PCAM 7, SPE",PHILADELPHIA,PA,19104,Allopathic & Osteopathic Physicians,Internal Medicine,Hematology & Oncology,Hematology & Oncology Physician
4,1598152944,2084N0400X,Psychiatry & Neurology,Neurology,330 S 9TH ST,None,PHILADELPHIA,PA,19107,Allopathic & Osteopathic Physicians,Psychiatry & Neurology,Neurology,Neurology Physician


In [15]:
def classify_preventative(row):

    classification = str(row["Classification"]).strip()
    specialization = str(row["Specialization"]).strip()
    grouping = str(row["Grouping"]).strip()

    primary_care = [
        "Family Medicine",
        "Internal Medicine",
        "General Practice",
        "Pediatrics",
        "Preventive Medicine"
    ]

    preventative_specialists = [
        "Cardiology",
        "Endocrinology",
        "Pulmonary Disease",
        "Nephrology",
        "Rheumatology",
        "Psychiatry",
        "Neurology",
        "Geriatric Medicine",
        "Obstetrics & Gynecology"
    ]

    outpatient_sites = [
        "Clinic/Center"
    ]

    non_preventative_facility = [
        "Skilled Nursing Facility",
        "Hospitalist",
        "Emergency Medicine",
        "Hospice and Palliative Medicine"
    ]

    if classification in primary_care:
        return "Primary Preventative Care"

    if classification == "Nurse Practitioner":
        return "Primary Preventative Care"

    if classification == "Physician Assistant":
        return "Primary Preventative Care"

    if classification in outpatient_sites:
        return "Primary Preventative Care"

    if classification in preventative_specialists:
        return "Preventative Specialist"

    if specialization in preventative_specialists:
        return "Preventative Specialist"

    if classification in non_preventative_facility:
        return "Non-Preventative / Facility"

    if grouping == "Nursing Care Providers":
        return "Non-Preventative / Facility"

    return "Other"

categories = []

for index, row in full_table.iterrows():
    category = classify_preventative(row)
    categories.append(category)

full_table["Preventative_Category"] = categories

full_table.head(30)

,NPI,Code,API_Category,API_Specialty,Address_1,Address_2,City,State,ZIP,Grouping,Classification,Specialization,Display Name,Preventative_Category
0,1790565554,314000000X,Skilled Nursing Facility,None,4401 HAVERFORD AVE,None,PHILADELPHIA,PA,19104,Nursing & Custodial Care Facilities,Skilled Nursing Facility,NaN,Skilled Nursing Facility,Non-Preventative / Facility
1,1255652087,207R00000X,Internal Medicine,None,3400 CIVIC CENTER BLVD,"DIVISION OF HEMATOLOGY-ONCOLOGY, PCAM 7, SPE",PHILADELPHIA,PA,19104,Allopathic & Osteopathic Physicians,Internal Medicine,NaN,Internal Medicine Physician,Primary Preventative Care
2,1255652087,390200000X,Student in an Organized Health Care Education/...,None,3400 CIVIC CENTER BLVD,"DIVISION OF HEMATOLOGY-ONCOLOGY, PCAM 7, SPE",PHILADELPHIA,PA,19104,"Student, Health Care",Student in an Organized Health Care Education/...,NaN,Student in an Organized Health Care Education/...,Other
3,1255652087,207RH0003X,Internal Medicine,Hematology & Oncology,3400 CIVIC CENTER BLVD,"DIVISION OF HEMATOLOGY-ONCOLOGY, PCAM 7, SPE",PHILADELPHIA,PA,19104,Allopathic & Osteopathic Physicians,Internal Medicine,Hematology & Oncology,Hematology & Oncology Physician,Primary Preventative Care
4,1598152944,2084N0400X,Psychiatry & Neurology,Neurology,330 S 9TH ST,None,PHILADELPHIA,PA,19107,Allopathic & Osteopathic Physicians,Psychiatry & Neurology,Neurology,Neurology Physician,Preventative Specialist
5,1346860350,261QM0855X,Clinic/Center,Adolescent and Children Mental Health,954 BELMONT AVE,None,PHILADELPHIA,PA,19104,Ambulatory Health Care Facilities,Clinic/Center,Adolescent and Children Mental Health,Adolescent and Children Mental Health Clinic/C...,Primary Preventative Care
6,1891923306,2080P0204X,Pediatrics,Pediatric Emergency Medicine,100 E PENN SQ,9TH FL,PHILADELPHIA,PA,19107,Allopathic & Osteopathic Physicians,Pediatrics,Pediatric Emergency Medicine,Pediatric Emergency Medicine (Pediatrics) Phys...,Primary Preventative Care
7,1558005157,390200000X,Student in an Organized Health Care Education/...,None,3600 SPRUCE STREET,"4TH FLOOR, MALONEY BUILDING",PHILADELPHIA,PA,19104,"Student, Health Care",Student in an Organized Health Care Education/...,NaN,Student in an Organized Health Care Education/...,Other
8,1558005157,390200000X,Student in an Organized Health Care Education/...,None,3600 SPRUCE STREET,"4TH FLOOR, MALONEY BUILDING",PHILADELPHIA,PA,19104,"Student, Health Care",Student in an Organized Health Care Education/...,NaN,Student in an Organized Health Care Education/...,Other
9,1396780789,208000000X,Pediatrics,None,100 E PENN SQ,9TH FLOOR,PHILADELPHIA,PA,19107,Allopathic & Osteopathic Physicians,Pediatrics,NaN,Pediatrics Physician,Primary Preventative Care


In [16]:
zip_summary = (
    full_table.groupby(["ZIP", "Preventative_Category"])
    .size()
    .reset_index(name="Provider_Count")
)

zip_summary.head(20)

,ZIP,Preventative_Category,Provider_Count
0,00652,Primary Preventative Care,1
1,02115,Primary Preventative Care,2
2,06288,Other,1
3,06510,Other,1
4,06510,Primary Preventative Care,2
5,08034,Other,1
6,08043,Primary Preventative Care,2
7,08108,Primary Preventative Care,2
8,08820,Primary Preventative Care,3
9,08902,Other,1
